# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *64*  
**Kaggle challenge:** *Classic*   
**Kaggle team name (exact):** "*Team Nah!*"  

**Author 1 (sciper):** Andrew Brown (370751)  
**Author 2 (sciper):** Nour Lachat (302397)   
**Author 3 (sciper):** Henry Farrell (402247) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

In [1]:
import src.lab1 as lab1 
import src.lab_01_utils as lab1utils
import src.lab2 as lab2
import src.lab_02_utils as lab2utils
import src.lab3 as lab3
import src.lab_03_utils as lab3utils
import src.preprocessing as pre
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from skimage.morphology import remove_small_objects, remove_small_holes, closing, disk, opening
from skimage.transform import rotate, resize
from sklearn.metrics.pairwise import euclidean_distances
from skimage.measure import regionprops
import cv2
from typing import Literal, Callable


In [2]:
# Load the Datasets
references_path = "../data/dataset_project_iapr2025/references"
reference_images_raw, reference_labels, reference_dict = pre.load_reference_images(references_path)


train_path = "../data/dataset_project_iapr2025/train"
train_images_raw, train_labels, train_dict = pre.load_train_images(train_path)

In [ ]:
# Print Shape of Images
print(f"Reference images shape: {reference_images_raw[0].shape}")
print(f"Train images shape: {train_images_raw[0].shape}")

In [4]:
reference_images = reference_images_raw.copy()
train_images = train_images_raw.copy()

# Downsample the images and update the reference_images list and dictionary
for i, imgs in enumerate(reference_images):
    # Downsample the image using grid method
    downsampled_grid = pre.downsample_image(
        imgs,
        method="grid",
        target_height=400,
        target_width=600  # Only one dimension needs specification
    )

    # Update the reference_images list with the downsampled images
    reference_images[i] = downsampled_grid
    reference_dict[reference_labels[i]] = downsampled_grid


# Do the same with the train images
for i, imgs in enumerate(train_images):
    # Downsample the image using grid method
    downsampled_grid = pre.downsample_image(
        imgs,
        method="grid",
        target_height=400,
        target_width=600  # Only one dimension needs specification
    )

    # Update the reference_images list with the downsampled images
    train_images[i] = downsampled_grid
    train_dict[train_labels[i]] = downsampled_grid

In [ ]:
# Print New Image Shape vs Old Image Shape
print(f"Old Image Shape: {train_images_raw[0].shape}")
print(f"New Image Shape: {train_images[0].shape}")


In [ ]:
pre.plot_reference_images(reference_images, reference_labels)

## Section 1: Pre-Processing and Background Removal
The goal here is to take every image and separate the background from the foreground. This will make future isoltion, cropping, and classification much easier.

In [7]:
# Plot the RGB histograms of the first N images in 3 subplots R vs G, R vs B, and G vs B

In [8]:
# Start by extracting the RGB data from all the images
N = len(train_labels)

for i in range(N):
    label = train_labels[i]
    img = train_dict[label]
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(img)
    H, S, V = lab1.extract_hsv_channels(img)
    
    # Update the dictionary to include the RGB channels
    train_dict[label] = {
        "image": img,  # Original image
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,   # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    }

In [ ]:
label = train_labels[0]
lab1utils.plot_colors_histo(
    img = train_dict[label]['image'],
    func = lab1.extract_rgb_channels,
    labels = ["Red", "Green", "Blue"],
)

In [ ]:
label = train_labels[0]
lab1utils.plot_colors_histo(
    img = train_dict[label]['image'],
    func = lab1.extract_hsv_channels,
    labels = ["Hue", "Saturation", "Value"],
)

In [ ]:
### HSV Thresholding ###

# Added an "HSV" key to the dictionary to story the HSV thresholded images

for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the HSV thresholding
    img_hsv = lab1.apply_hsv_threshold(img, H_min=0, H_max=0.2, S_min=0, S_max=1, V_min=0, V_max=0.65)
    
    # Add the new key-value pair to the dictionary
    dict["hsv_image"] = img_hsv

# Plot the original image and the thresholded image
label = train_labels[0]

# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['hsv_image'],
    title="Original vs Thresholded Image")
# Compare the original image and the thresholded image




In [ ]:


image  = train_dict[label]['hsv_image']
image_morph = apply_morphology(image)

# Plot the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['hsv_image'],
    image2=image_morph,
    title="Thresholded vs Morphology Image")

In [ ]:
# Convert the images to grayscale and add the grayscale images to the dictionary
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the HSV thresholding
    img_grey = pre.RGB2greyscale(img)
    # Add the new key-value pair to the dictionary
    dict["grey"] = img_grey

# Plot the original image and the thresholded image
label = train_labels[0]

# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['grey'],
    title="Original vs Grey Image")
# Compare the original image and the thresholded image



In [ ]:
# Load image
image = np.uint8(train_dict[label]['grey'])

t_lower = 40
t_upper = 85 
aperture_size = 3 
L2Gradient = False
cv_canny = cv2.Canny(image, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)

# Display results
# Display results using matplotlib
plt.figure(figsize=(10, 5))  # Set the figure size

# Plot the original image
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1
plt.imshow(image, cmap='gray')  # Display the image in grayscale
plt.title('Original Image')
plt.axis('off')  # Turn off axis

# Plot the Canny edge-detected image
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2
plt.imshow(cv_canny, cmap='gray')  # Display the Canny result in grayscale
plt.title('Canny Edge Detection')
plt.axis('off')  # Turn off axis

plt.tight_layout()  # Adjust spacing between subplots
plt.show()

In [ ]:
# Apply Canny edge detection to the images and add the Canny images to the dictionary
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = np.uint8(dict["grey"])

    # Define the parameters for Canny edge detection
    t_lower = 40
    t_upper = 85 
    aperture_size = 3 
    L2Gradient = False
    
    # Apply the Canny edge detection
    img_canny = cv2.Canny(img, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)
    
    # Add the new key-value pair to the dictionary
    dict["canny"] = img_canny

# Apply morphology to the Canny edge-detected image
for dict in train_dict.values():
    # Extract the image from the dictionary
    img = dict["canny"]
    
    # Apply the morphology
    img_morph = lab1.apply_morphology(img)
    
    # Add the new key-value pair to the dictionary
    dict["canny_morph"] = img_morph

# Plot the original image and the thresholded image
label = train_labels[6]
# Compare the original image and the thresholded image
pre.compare_2_images(
    image1=train_dict[label]['image'],
    image2=train_dict[label]['canny_morph'],
    title="Canny vs Morphology Image")


In [ ]:
### Do all of the same previous operations to th reference images to find good countours ###

# Canny Edge Detection Params
t_lower = 40
t_upper = 85 
aperture_size = 3 
L2Gradient = False

#Morphology Params
kernel_size=4
min_area_th=500
min_obj_size=250
connect=1

# Start by extracting the RGB data from all the images
N = len(reference_labels)

for i in range(N):
    label = reference_labels[i]
    img = reference_dict[label]
    # Extract the RGB channels
    R, G, B = lab1.extract_rgb_channels(img)
    H, S, V = lab1.extract_hsv_channels(img)
    
    # Update the dictionary to include the RGB channels
    reference_dict[label] = {
        "image": img,  # Original image
        "R": R,  # Red channel
        "G": G,  # Green channel
        "B": B,   # Blue channel
        "H": H,  # Hue channel
        "S": S,  # Saturation channel
        "V": V   # Value channel
    }

# Convert the images to grayscale and add the grayscale images to the dictionary
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = dict["image"]
    
    # Apply the HSV thresholding
    img_grey = pre.RGB2greyscale(img)
    # Add the new key-value pair to the dictionary
    dict["grey"] = img_grey

# Apply Canny edge detection to the images and add the Canny images to the dictionary
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = np.uint8(dict["grey"])
    
    # Apply the Canny edge detection
    img_canny = cv2.Canny(img, t_lower, t_upper, apertureSize=aperture_size, L2gradient=L2Gradient)
    
    # Add the new key-value pair to the dictionary
    dict["canny"] = img_canny

# Apply morphology to the Canny edge-detected image
for dict in reference_dict.values():
    # Extract the image from the dictionary
    img = dict["canny"]
    
    # Apply the morphology
    img_morph = lab1.apply_morphology(img, kernel_size=kernel_size, min_area_th=min_area_th, min_obj_size=min_obj_size, connect=connect)
    
    # Add the new key-value pair to the dictionary
    dict["canny_morph"] = img_morph

In [ ]:
import math
import matplotlib.pyplot as plt

def plot_ref_images_vs_filtered(reference_dict, reference_labels):
    """
    Plot the original and filtered images for each reference image.
    Args:
        reference_dict (dict): Dictionary containing reference images and their properties.
        reference_labels (list): List of labels corresponding to the reference images.
    """

    # Calculate the number of rows needed for 3 columns
    num_rows = math.ceil(len(reference_labels))

    # Create a figure with subplots
    fig, axes = plt.subplots(num_rows, 3, figsize=(15, 6 * num_rows))
    fig.suptitle("Filtered vs Original Images", fontsize=16)

    # Flatten the axes array for easier indexing
    axes = axes.flatten()

    for i, label in enumerate(reference_labels):
        # Original image (first column)
        original_image = reference_dict[label]['image']
        axes[i * 3].imshow(original_image)
        axes[i * 3].set_title(f"Original: {label}")
        axes[i * 3].axis('off')

        # Filtered image (second column)
        filtered_image = reference_dict[label]['canny']
        axes[i * 3 + 1].imshow(filtered_image, cmap='gray')
        axes[i * 3 + 1].set_title(f"Filtered: {label}")
        axes[i * 3 + 1].axis('off')

        # Another filtered image (third column)
        filtered_image_morph = reference_dict[label]['canny_morph']
        axes[i * 3 + 2].imshow(filtered_image_morph, cmap='gray')
        axes[i * 3 + 2].set_title(f"Filtered Morph: {label}")
        axes[i * 3 + 2].axis('off')

    # Hide any unused subplots
    for j in range(len(reference_labels) * 3, len(axes)):
        axes[j].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Call the function to plot the images
plot_ref_images_vs_filtered(reference_dict, reference_labels)

In [64]:
def find_contour(image: np.ndarray):
    """
    Find the contours for a single image.

    Args
    ----
    image: np.ndarray (H, W)
        Source image to process (binary image).

    Return
    ------
    contours: list of np.ndarray
        List of arrays containing the coordinates of the contours. Each element of the 
        list is an array of 2D coordinates (K, 2) where K depends on the number of elements 
        that form the contour.
    """
    # Ensure the input image is binary
    binary_image = (image > 0).astype(np.uint8)

    # Find contours using OpenCV
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Convert contours to a list of arrays with shape (K, 2)
    contours = [contour.squeeze() for contour in contours if contour.size > 0]

    return contours

def simple_segmentation(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Thresholds for neutral/light backgrounds — adjust as needed
    lower = np.array([0, 0, 0])
    upper = np.array([100, 10, 180])

    # Create binary mask: 0 for background, 255 for foreground
    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Optional: morphological cleanup
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    return cleaned

In [ ]:
# Load image
image = reference_images[0]

# Create mask
mask = simple_segmentation(image)

# Create a blue background (same shape as image)
blue_background = np.full_like(image, (255, 0, 0))  # BGR for blue

# Combine chocolates with blue background
foreground = cv2.bitwise_and(image, image, mask=mask)
background = cv2.bitwise_and(blue_background, blue_background, mask=cv2.bitwise_not(mask))
result = cv2.add(foreground, background)

# Convert to RGB for display
original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

# Display side-by-side
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(original_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(result_rgb)
plt.title("Background Replaced with Blue")
plt.axis("off")

plt.tight_layout()
plt.show()
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [30]:
def get_binary_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Threshold for neutral/light backgrounds
    lower = np.array([0, 0, 160])
    upper = np.array([180, 40, 255])

    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Morphological cleanup (optional)
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    # Convert to binary: 0 = foreground (chocolates), 1 = background
    binary_image = np.where(cleaned > 0, 0, 1).astype(np.uint8)

    return binary_image


In [ ]:
binary_mask = get_binary_mask(image)

# Display binary mask
plt.imshow(binary_mask, cmap='gray')
plt.title("Binary Image (0=Object, 1=Background)")
plt.axis('off')
plt.show()
mask = simple_segmentation(np.uint8(image))
result = cv2.bitwise_and(image, image, mask=mask)
# Convert to grayscale and preprocess (e.g., thresholding)
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, binary_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)

# Find contours
contours = find_contour(binary_image)

# Draw contours in red on the original image
output_image = image.copy()
cv2.drawContours(output_image, contours, -1, (0, 0, 255), 2)  # Red color (BGR: (0, 0, 255)), thickness=2

# Display the result
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for matplotlib
plt.title("Contours in Red")
plt.axis('off')
plt.show()


In [ ]:

# image_folder=train_path
# # Loop through filenames from L1000777.jpg to L1000790.jpg
# for i in range(1000757, 1000758):
#     filename = f'L{i}.JPG'
#     img_path = os.path.join(image_folder, filename)
    
#     if os.path.exists(img_path):
#         img = cv2.imread(img_path)
#         img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB for matplotlib
#         plt.figure(figsize=(4, 4))
#         plt.imshow(img)
#         plt.title(filename)
#         plt.axis('off')
#         plt.show()
#     else:
#         print(f"Image {filename} not found.")
# def find_contour(images: np.ndarray):
#     """
#     Find the contours for the set of images
    
#     Args
#     ----
#     images: np.ndarray (N, 28, 28)
#         Source images to process

#     Return
#     ------
#     contours: list of np.ndarray
#         List of N arrays containing the coordinates of the contour. Each element of the 
#         list is an array of 2d coordinates (K, 2) where K depends on the number of elements 
#         that form the contour. 
#     """

#     # Get number of images to process
#     N, _, _ = np.shape(images)
#     # Fill in dummy values (fake points)
#     contours = [np.array([[0, 0], [1, 1]]) for i in range(N)]
#     contour= [np.array([[0, 0], [1, 1]]) for i in range(N)]

#     # ------------------
#     for i in range(N):
#         contours_img=cv2.findContours(images[i], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
#         max_length = 0
#         longest_contour = None

#         for j in range(len(contours_img[0])):
#             length=len(contours_img[0][j])
#             if length > max_length:
#                 longest_contour = contours_img[0][j]
#                 max_length = length
#         contours[i] = longest_contour

#         if len(contours_img[0]) > 0:
#             for j in range(len(contours_img[0])):
#                 length = len(contours_img[0][j])
#                 if length > max_length:
#                     longest_contour = contours_img[0][j]
#                     max_length = length
            
#             # Only update contours[i] and squeeze if a contour was found
#             if longest_contour is not None:
#                 contours[i] = longest_contour
#                 contours[i] = contours[i].squeeze()  # Now shape is (K, 2)

#         if longest_contour is not None:
#             contours[i] = longest_contour
#             contours[i] = contours[i].squeeze()  # Now shape is (K, 2)
#         else:

#             print(f"Warning: No valid contour found for image {i}")
#       # ------------------
#     return contours









